# S&P 500 Tail Risk Challenge

You are a risk analyst at an investment fund. Your team uses a normal distribution as a baseline model for daily stock returns. Your task is to identify the S&P 500 stock whose historical returns exhibit the strongest combination of **high excess kurtosis and negative skewness**.

High kurtosis highlights the influence of extreme observations, while negative skewness indicates asymmetry towards losses. This combination may raise concerns about risk estimates based on normality.

### Your objective

Find the eligible stock with the **highest combined tail-risk score** over **2 January 2024 to 31 December 2025**, inclusive, using **daily Adjusted Close prices**, adjusted for stock splits and dividends.

Calculate sample skewness and sample excess kurtosis using finite-sample bias corrections. The combined score assigns:

- **50% weight** to the percentile rank of excess kurtosis.
- **50% weight** to the percentile rank of negative skewness, so more negative skewness receives a higher score.

Percentile ranks are calculated among eligible stocks as average rank divided by the number of eligible stocks. Tied values receive the same average rank.

### Eligible stocks

Use the tickers in Wikipedia’s S&P 500 component stocks table, based on the snapshot downloaded on 7 October 2026. Later changes to the list do not affect this competition.

A stock is eligible only if:

- Its skewness is **strictly negative**.
- Valid daily returns are available for **at least 90% of the US equity trading sessions** in the analysis period.
- ts skewness and kurtosis are finite and defined.

Weekends and market holidays are excluded from the coverage denominator. All answers are evaluated using the instructor’s fixed reference dataset.

### Schedule and attempts

- **Opens**: Thursday, 8 October 2026, at 14:00.
- **Closes**: Monday, 12 October 2026, at 23:59.
- **Time zone**: Europe/Madrid.
- **Attempts allowed**: 5.
- **Final grade**: Your highest grade across all attempts.

Every submitted attempt counts, including repeated, incorrectly formatted or ineligible tickers.

### How to submit

Enter **only the exact Wikipedia ticker**, in uppercase and preserving punctuation. For example, enter BRK.B, not BRK-B. Do not enter company names, explanations or multiple tickers.

Select **“Submit all and finish”** to confirm your answer.

### Feedback and grading

After submitting an eligible ticker, Moodle will display its excess kurtosis, skewness, combined score, distance from the best combined score, and grade.

The exact reference grade is calculated relative to the highest combined score. Moodle grades are rounded down to supported grading percentages, while the feedback preserves the numerical reference results. Only optimal answers receive **10/10**. Unrecognised or ineligible tickers receive **0/10**.

To check your previous attempts, select **“Finish review”** and return to the questionnaire’s main page.

### Financial interpretation

This score identifies a historical return profile that deserves closer risk analysis. It does not directly measure potential losses, establish that an asset is the riskiest investment, or predict future performance.

In [1]:
import numpy as np
import pandas as pd
import requests
from io import StringIO
import yfinance as yf

In [2]:


# Download the S&P 500 component table.
url = "https://en.wikipedia.org/wiki/List_of_S%26P_500_companies"
response = requests.get(url, headers={"User-Agent": "Mozilla/5.0"}, timeout=30)
response.raise_for_status()

# Preserve Wikipedia symbols and adapt them for Yahoo Finance.
wikipedia_tickers = pd.read_html(StringIO(response.text))[0]["Symbol"].tolist()
sp500_tickers = [ticker.replace(".", "-") for ticker in wikipedia_tickers]

sp500_tickers

['MMM',
 'AOS',
 'ABT',
 'ABBV',
 'ACN',
 'ADBE',
 'AMD',
 'AES',
 'AFL',
 'A',
 'APD',
 'ABNB',
 'AKAM',
 'ALB',
 'ARE',
 'ALGN',
 'ALLE',
 'LNT',
 'ALL',
 'GOOGL',
 'GOOG',
 'MO',
 'AMZN',
 'AMCR',
 'AEE',
 'AEP',
 'AXP',
 'AIG',
 'AMT',
 'AWK',
 'AMP',
 'AME',
 'AMGN',
 'APH',
 'ADI',
 'AON',
 'APA',
 'APO',
 'AAPL',
 'AMAT',
 'APP',
 'APTV',
 'ACGL',
 'ADM',
 'ARES',
 'ANET',
 'AJG',
 'AIZ',
 'T',
 'ATO',
 'ADSK',
 'ADP',
 'AZO',
 'AVY',
 'AXON',
 'BKR',
 'BALL',
 'BAC',
 'BAX',
 'BDX',
 'BRK-B',
 'BBY',
 'TECH',
 'BIIB',
 'BLK',
 'BX',
 'XYZ',
 'BE',
 'BNY',
 'BA',
 'BKNG',
 'BSX',
 'BMY',
 'AVGO',
 'BR',
 'BRO',
 'BF-B',
 'BG',
 'BXP',
 'CHRW',
 'CDNS',
 'CPT',
 'COF',
 'CAH',
 'CCL',
 'CARR',
 'CVNA',
 'CASY',
 'CAT',
 'CBOE',
 'CBRE',
 'CDW',
 'COR',
 'CNC',
 'CNP',
 'CF',
 'CRL',
 'SCHW',
 'CHTR',
 'CVX',
 'CMG',
 'CB',
 'CHD',
 'CIEN',
 'CI',
 'CINF',
 'CTAS',
 'CSCO',
 'C',
 'CFG',
 'CLX',
 'CME',
 'CMS',
 'KO',
 'CTSH',
 'COHR',
 'COIN',
 'CL',
 'CMCSA',
 'FIX',
 'COP',
 'E

### 1. Download adjusted closing prices

Download daily adjusted closing prices for all tickers in `sp500_tickers`, from 2 January 2024 to 31 December 2025, inclusive. These prices account for stock splits and dividends.

The end date in `yfinance` is exclusive, so we use 1 January 2026 to include the final day of the analysis period.

In [3]:

prices = yf.download(
    sp500_tickers,
    start="2024-01-02",
    end="2026-01-01",
    auto_adjust=False,
)["Adj Close"]

prices.head()

[***                    6%                       ]  28 of 503 completed$VYLR: Data doesn't exist for startDate = 1704171600, endDate = 1767243600
[****************      34%                       ]  170 of 503 completed$HONA: Data doesn't exist for startDate = 1704171600, endDate = 1767243600
[**********************71%*********              ]  358 of 503 completed$FDXF: Data doesn't exist for startDate = 1704171600, endDate = 1767243600
[*********************100%***********************]  501 of 503 completed$HUM: possibly delisted; no price data found  (1d 2024-01-02 -> 2026-01-01)
[*********************100%***********************]  503 of 503 completed

4 Failed downloads:
['VYLR', 'HONA', 'FDXF']: Data doesn't exist for startDate = 1704171600, endDate = 1767243600
['HUM']: possibly delisted; no price data found  (1d 2024-01-02 -> 2026-01-01)


Ticker,A,AAPL,ABBV,ABNB,ABT,ACGL,ACN,ADBE,ADI,ADM,...,WY,WYNN,XEL,XOM,XYL,XYZ,YUM,ZBH,ZBRA,ZTS
Date,,,,,,,,,,,,,,,,,,,,,
2024-01-02,135.917877,183.404022,145.533890,134.479996,103.769417,71.840385,329.828674,580.070007,185.602814,66.223297,...,31.840797,92.002754,57.849808,93.601593,109.875603,72.220001,122.321892,118.228027,267.980011,189.126083
2024-01-03,128.482788,182.030746,146.116669,133.419998,103.457680,73.162132,321.272095,571.789978,181.173203,66.560059,...,31.172678,91.448402,57.950073,94.388016,107.836029,68.629997,122.388268,117.331978,252.520004,185.623901
2024-01-04,128.326050,179.718933,147.027328,133.720001,104.836868,73.666107,320.482941,567.049988,178.402283,65.304039,...,30.742521,91.623474,58.086788,93.565025,108.580330,68.150002,121.999580,117.069023,252.970001,186.691910
2024-01-05,127.895027,178.997742,147.646515,135.979996,104.666847,73.038513,320.036133,564.599976,178.862518,64.448486,...,30.824886,93.023956,58.086788,93.848503,108.406349,66.959999,121.667763,116.854759,252.690002,187.471191
2024-01-08,130.657471,183.324982,146.999954,140.080002,106.178276,73.333298,323.582336,580.549988,181.192352,64.676018,...,31.163527,94.385521,57.950073,92.284821,109.082970,69.309998,121.952164,118.792915,261.089996,188.721970


### 2. Calculate daily logarithmic returns

Calculate daily logarithmic returns from adjusted closing prices:

$$
r_t = \log(P_t) - \log(P_{t-1})
$$

The first observation has no return because there is no preceding price. Missing prices are left unfilled, so both adjacent returns are missing.

In [4]:
log_returns = np.log(prices).diff()

log_returns.head()

Ticker,A,AAPL,ABBV,ABNB,ABT,ACGL,ACN,ADBE,ADI,ADM,...,WY,WYNN,XEL,XOM,XYL,XYZ,YUM,ZBH,ZBRA,ZTS
Date,,,,,,,,,,,,,,,,,,,,,
2024-01-02,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2024-01-03,-0.056256,-0.007516,0.003996,-0.007913,-0.003009,0.018231,-0.026285,-0.014377,-0.024155,0.005072,...,-0.021206,-0.006044,0.001732,0.008367,-0.018737,-0.050987,0.000542,-0.007608,-0.059422,-0.018691
2024-01-04,-0.001221,-0.012781,0.006213,0.002246,0.013243,0.006865,-0.002459,-0.008324,-0.015412,-0.019051,...,-0.013895,0.001913,0.002356,-0.008757,0.006878,-0.007019,-0.003181,-0.002244,0.001780,0.005737
2024-01-05,-0.003364,-0.004021,0.004203,0.016760,-0.001623,-0.008556,-0.001395,-0.004330,0.002576,-0.013188,...,0.002676,0.015170,0.000000,0.003025,-0.001604,-0.017616,-0.002724,-0.001832,-0.001107,0.004165
2024-01-08,0.021369,0.023887,-0.004389,0.029706,0.014337,0.004028,0.011020,0.027858,0.012942,0.003524,...,0.010926,0.014531,-0.002356,-0.016802,0.006222,0.034494,0.002335,0.016450,0.032702,0.006650


### 3. Filter stocks by return coverage

Keep stocks with valid daily returns for at least 90% of the trading sessions in the analysis period.

Assume the price index contains all US equity trading sessions in the period. The first missing return counts towards the missing-data allowance.

In [5]:
# Keep stocks with at most 10% missing returns.
eligible_tickers = log_returns.isna().mean() <= 0.10
log_returns = log_returns.loc[:, eligible_tickers]

print(f"Stocks retained: {log_returns.shape[1]}")


Stocks retained: 494


### 4. Calculate skewness and excess kurtosis

Calculate sample skewness and excess kurtosis for each stock using finite-sample bias corrections.

Keep only stocks with strictly negative skewness and finite, defined statistics. Pandas excludes missing returns when calculating these statistics.

In [6]:
statistics = pd.DataFrame({
    "skewness": log_returns.skew(),
    "excess_kurtosis": log_returns.kurt(),
})

# Keep stocks with negative skewness.
statistics = statistics.loc[statistics["skewness"] < 0]

statistics.head()

,skewness,excess_kurtosis
Ticker,,
A,-0.159556,3.987875
ABBV,-1.584740,12.866807
ABNB,-0.108125,9.349224
ABT,-0.656516,6.165389
ACGL,-0.621664,4.330212


### 5. Calculate the combined score and identify the winner

Calculate percentile ranks among eligible stocks as average rank divided by the number of eligible stocks. Higher excess kurtosis and more negative skewness receive higher ranks.

Combine both percentiles with equal weights:

$$
C_i = 0.5\,P(K_i) + 0.5\,P(-S_i)
$$

Sort stocks by score and identify all winners, including ties. Convert the winning Yahoo Finance symbols back to their original Wikipedia symbols.

In [7]:
# Calculate percentile ranks among eligible stocks.
statistics["score"] = (
    0.5 * statistics["excess_kurtosis"].rank(method="average", pct=True)
    + 0.5 * (-statistics["skewness"]).rank(method="average", pct=True)
)

statistics = statistics.sort_values("score", ascending=False)

# Restore the original Wikipedia symbols.
ticker_mapping = dict(zip(sp500_tickers, wikipedia_tickers))
statistics.index = statistics.index.map(ticker_mapping)

display(statistics.head(10))

# Display all stocks tied for the highest score.
winners = statistics.loc[statistics["score"] == statistics["score"].max()]
display(winners)

,skewness,excess_kurtosis,score
Ticker,,,
GL,-14.182726,280.436909,1.000000
FISV,-11.860914,195.786590,0.996970
EW,-8.176504,138.983335,0.993939
CNC,-7.950345,125.121726,0.990909
DXCM,-6.834688,101.375975,0.987879
ALGN,-6.128305,91.389379,0.983333
WST,-5.990204,97.098503,0.981818
IT,-6.067746,85.554512,0.980303
SNPS,-5.348775,75.089351,0.974242


,skewness,excess_kurtosis,score
Ticker,,,
GL,-14.182726,280.436909,1.0
